<a href="https://colab.research.google.com/github/Taginoyoshi/scholarSafeMotorClassifV2/blob/main/ebcaIFCEdatasetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import shap
from scipy.sparse import issparse, hstack, vstack
from scipy.stats import ks_2samp

from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, log_loss
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Motores de Boosting configurados como Florestas
from lightgbm import LGBMClassifier
from xgboost import XGBRFClassifier

# PSO para Otimização
import pyswarms as ps

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

VERBOSE = False

def load_and_clean_data(filepath='ifce_telecom_unificado - ifce_telecom_unificado.csv', verbose=True):
    if verbose: print(f"[1/11] Carregando dataset bruto para treinamento: {filepath}")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        if verbose: print(f"[ERRO] Arquivo '{filepath}' não encontrado.")
        raise

    if 'Desc_Sit_Matricula' not in df.columns:
        raise ValueError("Coluna 'Desc_Sit_Matricula' não encontrada. Use a base bruta original.")

    status_map = {
        'Formado': 'CONCLUÍDO',
        'Abandono': 'CANCELADO',
        'Cancelado Voluntariamente': 'CANCELADO',
        'Cancelado Compulsoriamente': 'CANCELADO'
    }
    df['status'] = df['Desc_Sit_Matricula'].map(status_map)
    df = df.dropna(subset=['status'])

    if verbose: print("      > Pivotando histórico: agrupando disciplinas em colunas para cada aluno...")
    subject_cols = ['DESC_DISCIPLINA', 'CARGA_HOR', 'Sit_Disciplina', 'N_FALTAS', 'NOTA', 'Equivalencia']
    student_cols = [c for c in df.columns if c not in subject_cols]

    df_student = df[student_cols].drop_duplicates(subset=['Cod_Pessoa'])
    pivot_df = df.pivot_table(index='Cod_Pessoa', columns='DESC_DISCIPLINA', values='NOTA', aggfunc='max')
    df_merged = df_student.merge(pivot_df, on='Cod_Pessoa', how='inner')

    cols_to_drop_init = ['Cod_Pessoa', 'Desc_Sit_Matricula']
    df_merged = df_merged.drop(columns=[c for c in cols_to_drop_init if c in df_merged.columns])

    if verbose: print("      > Calculando Informação Mútua (MI) para remover Vazamentos de Dados...")
    df_mi = df_merged.copy()
    for col in df_mi.columns:
        if df_mi[col].dtype == 'object':
            df_mi[col] = LabelEncoder().fit_transform(df_mi[col].astype(str))
        else:
            df_mi[col] = df_mi[col].fillna(-1)

    X_mi = df_mi.drop(columns=['status'])
    y_mi = LabelEncoder().fit_transform(df_mi['status'])

    mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
    mi_series = pd.Series(mi_scores, index=X_mi.columns).sort_values(ascending=False)

    threshold = 0.30
    leakage_cols = mi_series[mi_series > threshold].index.tolist()

    if verbose:
        print(f"      > ATENÇÃO: {len(leakage_cols)} colunas removidas automaticamente por vazamento (MI > {threshold}).")
        for col, score in mi_series.head(7).items():
            if score > threshold:
                print(f"        - {col}: {score:.4f}")

    df_clean = df_merged.drop(columns=leakage_cols)

    subject_names = pivot_df.columns.tolist()
    for col in df_clean.select_dtypes(include='number').columns:
        if col in subject_names:
            df_clean[col] = df_clean[col].fillna(0)
        else:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    for col in df_clean.select_dtypes(include='object').columns:
        df_clean[col] = df_clean[col].fillna('Não Informado')

    return df_clean, subject_names, leakage_cols

def preprocess_sparse(df, verbose=True):
    if verbose: print("[2/11] Pré-processamento com matrizes esparsas e Pipelines...")
    y = df['status']
    X = df.drop(columns=['status'])
    expected_features = X.columns.tolist()

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    cat_cols = X.select_dtypes(include=['object', 'category']).columns
    num_cols = X.select_dtypes(include=['number']).columns

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MaxAbsScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Não Informado')),
        ('encoder', OneHotEncoder(sparse_output=True, handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_cols),
            ('cat', cat_pipeline, cat_cols)
        ],
        remainder='drop'
    )

    X_processed = preprocessor.fit_transform(X)
    return X_processed, y_encoded, le, preprocessor, expected_features

def extended_balance_cascade(X, y, T=5, imbalance_ratio=2.5, verbose=False):
    classes, counts = np.unique(y, return_counts=True)
    min_class = classes[np.argmin(counts)]
    maj_class = classes[np.argmax(counts)]

    idx_min = np.where(y == min_class)[0]
    idx_maj = np.where(y == maj_class)[0]

    P_X, P_y = X[idx_min], y[idx_min]
    N_X, N_y = X[idx_maj], y[idx_maj]

    subsets = []

    for i in range(T):
        if len(idx_maj) == 0 or (len(idx_maj) / len(idx_min)) < imbalance_ratio:
            break

        sample_idx = np.random.choice(len(idx_maj), size=len(idx_min), replace=False)
        N_i_X = N_X[sample_idx]
        N_i_y = N_y[sample_idx]

        if issparse(P_X):
            X_train_c = vstack([P_X, N_i_X])
        else:
            X_train_c = np.vstack([P_X, N_i_X])
        y_train_c = np.concatenate([P_y, N_i_y])

        clf = DecisionTreeClassifier(max_depth=3, random_state=42+i)
        clf.fit(X_train_c, y_train_c)

        preds = clf.predict(N_X)
        correctly_classified_maj = (preds == maj_class)

        N_X = N_X[~correctly_classified_maj]
        N_y = N_y[~correctly_classified_maj]
        idx_maj = np.arange(len(N_y))

        if issparse(P_X):
            S_i_X = vstack([P_X, N_X])
        else:
            S_i_X = np.vstack([P_X, N_X])
        S_i_y = np.concatenate([P_y, N_y])

        subsets.append((S_i_X, S_i_y))

    if not subsets:
        if issparse(P_X):
            subsets.append((vstack([P_X, N_X]), np.concatenate([P_y, N_y])))
        else:
            subsets.append((np.vstack([P_X, N_X]), np.concatenate([P_y, N_y])))

    if verbose: print(f"      > EBCA gerou {len(subsets)} subconjunto(s) validado(s).")
    return subsets

class EBCADeepCascadeForest(BaseEstimator, ClassifierMixin):
    def __init__(self, max_layers=5, k_fold=3, T=3, imbalance_ratio=2.5,
                 lgbm_num_leaves=127, xgb_max_depth=10, random_state=42):
        self.max_layers = max_layers
        self.k_fold = k_fold
        self.T = T
        self.imbalance_ratio = imbalance_ratio
        self.lgbm_num_leaves = lgbm_num_leaves
        self.xgb_max_depth = xgb_max_depth
        self.random_state = random_state
        self.ebca_estimators_ = []
        self.layers_ = []
        self.best_layer_num_ = 0

    def _get_ebca_estimators(self, idx):
        lgbm = LGBMClassifier(boosting_type='rf', num_leaves=max(31, self.lgbm_num_leaves//2), bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, n_estimators=100, random_state=self.random_state + idx, n_jobs=-1, verbose=-1)
        xgb = XGBRFClassifier(n_estimators=100, max_depth=max(3, self.xgb_max_depth-2), subsample=0.8, colsample_bynode=0.8, random_state=self.random_state + idx + 100, n_jobs=-1, eval_metric='logloss')
        return lgbm, xgb

    def _get_layer_estimators(self, layer_idx):
        lgbm_rf = LGBMClassifier(boosting_type='rf', num_leaves=self.lgbm_num_leaves, n_estimators=100, bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, n_jobs=-1, verbose=-1, random_state=self.random_state + layer_idx * 10 + 1)
        xgb_rf = XGBRFClassifier(n_estimators=100, max_depth=self.xgb_max_depth, subsample=0.8, colsample_bynode=0.8, n_jobs=-1, eval_metric='logloss', random_state=self.random_state + layer_idx * 10 + 2)
        lgbm_et = LGBMClassifier(boosting_type='rf', num_leaves=self.lgbm_num_leaves, n_estimators=100, bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.4, extra_trees=True, n_jobs=-1, verbose=-1, random_state=self.random_state + layer_idx * 10 + 3)
        xgb_et = XGBRFClassifier(n_estimators=100, max_depth=self.xgb_max_depth, subsample=0.8, colsample_bynode=0.4, n_jobs=-1, eval_metric='logloss', random_state=self.random_state + layer_idx * 10 + 4)
        return [lgbm_rf, xgb_rf, lgbm_et, xgb_et]

    def fit(self, X, y, verbose=False):
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=self.random_state, stratify=y)

        if verbose: print(f"      > [Camada 0] Extraindo características via EBCA (T={self.T})...")
        subsets = extended_balance_cascade(X_train, y_train, T=self.T, imbalance_ratio=self.imbalance_ratio, verbose=verbose)
        self.ebca_estimators_ = []

        train_ebca_probas = []
        val_ebca_probas = []

        for i, (X_sub, y_sub) in enumerate(subsets):
            lgbm, xgb = self._get_ebca_estimators(i)
            lgbm.fit(X_sub, y_sub)
            xgb.fit(X_sub, y_sub)
            self.ebca_estimators_.extend([lgbm, xgb])
            train_ebca_probas.extend([lgbm.predict_proba(X_train), xgb.predict_proba(X_train)])
            val_ebca_probas.extend([lgbm.predict_proba(X_val), xgb.predict_proba(X_val)])

        train_ebca_concat = np.hstack(train_ebca_probas)
        val_ebca_concat = np.hstack(val_ebca_probas)

        X_train_current = hstack([X_train, train_ebca_concat]) if issparse(X_train) else np.hstack([X_train, train_ebca_concat])
        X_val_current = hstack([X_val, val_ebca_concat]) if issparse(X_val) else np.hstack([X_val, val_ebca_concat])

        if verbose: print(f"      > [Camadas 1..N] Iniciando Cascata Profunda Híbrida...")
        self.layers_ = []
        best_val_score = -np.inf

        for layer_idx in range(self.max_layers):
            estimators = self._get_layer_estimators(layer_idx)
            cv = KFold(n_splits=self.k_fold, shuffle=True, random_state=self.random_state)

            layer_train_probas = []
            layer_val_probas = []

            for est in estimators:
                oof_probas = cross_val_predict(est, X_train_current, y_train, cv=cv, method='predict_proba', n_jobs=-1)
                layer_train_probas.append(oof_probas)
                est.fit(X_train_current, y_train)
                layer_val_probas.append(est.predict_proba(X_val_current))

            avg_val_probas = np.mean(layer_val_probas, axis=0)
            current_val_score = roc_auc_score(y_val, avg_val_probas[:, 1])

            if verbose: print(f"        - Camada {layer_idx + 1}: AUC de Validação = {current_val_score:.4f}")

            if current_val_score <= best_val_score:
                if verbose: print("      > O desempenho parou de melhorar. Parada antecipada ativada.")
                break

            best_val_score = current_val_score
            self.layers_.append(estimators)
            self.best_layer_num_ = layer_idx + 1

            train_probas_concat = np.hstack(layer_train_probas)
            val_probas_concat = np.hstack(layer_val_probas)

            if issparse(X_train_current):
                X_train_current = hstack([X_train_current, train_probas_concat])
                X_val_current = hstack([X_val_current, val_probas_concat])
            else:
                X_train_current = np.hstack([X_train_current, train_probas_concat])
                X_val_current = np.hstack([X_val_current, val_probas_concat])

        if verbose: print(f"      > Treinamento finalizado com sucesso na Camada {self.best_layer_num_}.")
        return self

    def transform_to_last_layer(self, X):
        X_current = X.copy() if not issparse(X) else X.copy()
        ebca_probas = [est.predict_proba(X_current) for est in self.ebca_estimators_]
        probas_concat = np.hstack(ebca_probas)
        X_current = hstack([X_current, probas_concat]) if issparse(X_current) else np.hstack([X_current, probas_concat])

        for layer_idx in range(len(self.layers_) - 1):
            layer_probas = [est.predict_proba(X_current) for est in self.layers_[layer_idx]]
            probas_concat = np.hstack(layer_probas)
            X_current = hstack([X_current, probas_concat]) if issparse(X_current) else np.hstack([X_current, probas_concat])

        return X_current.tocsr() if issparse(X_current) else X_current

    def get_feature_names_for_last_layer(self, base_names):
        names = list(base_names)
        for t in range(len(self.ebca_estimators_) // 2):
            names.extend([f"EBCA_Sub{t+1}_LGBM_Class0", f"EBCA_Sub{t+1}_LGBM_Class1"])
            names.extend([f"EBCA_Sub{t+1}_XGB_Class0", f"EBCA_Sub{t+1}_XGB_Class1"])

        for l in range(len(self.layers_) - 1):
            names.extend([f"L{l+1}_LGBM_RF_Class0", f"L{l+1}_LGBM_RF_Class1"])
            names.extend([f"L{l+1}_XGB_RF_Class0", f"L{l+1}_XGB_RF_Class1"])
            names.extend([f"L{l+1}_LGBM_ET_Class0", f"L{l+1}_LGBM_ET_Class1"])
            names.extend([f"L{l+1}_XGB_ET_Class0", f"L{l+1}_XGB_ET_Class1"])
        return names

    def predict_proba(self, X):
        X_current = self.transform_to_last_layer(X)
        last_layer_probas = [est.predict_proba(X_current) for est in self.layers_[-1]]
        return np.mean(last_layer_probas, axis=0)

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

# --- FUNÇÃO DE OTIMIZAÇÃO GLOBAL (PSO MULTIPARÂMETROS) ---
def optimize_hyperparams(params, X_tr, y_tr, X_v, y_v):
    """ Otimiza o T, as Folhas do LightGBM e a Profundidade do XGBoost """
    scores = []
    for param in params:
        t_val = max(1, int(round(param[0])))
        lgbm_leaves = max(31, int(round(param[1])))
        xgb_depth = max(3, int(round(param[2])))

        model = EBCADeepCascadeForest(max_layers=2, k_fold=3, T=t_val,
                                      lgbm_num_leaves=lgbm_leaves, xgb_max_depth=xgb_depth, random_state=42)
        model.fit(X_tr, y_tr, verbose=False)
        preds = model.predict_proba(X_v)[:, 1]
        auc = roc_auc_score(y_v, preds)
        scores.append(1.0 - auc)
    return np.array(scores)

# --- AVALIAÇÃO E MÉTRICAS ---
def calculate_paper_metrics(y_true, y_pred, y_pred_proba):
    f_measure = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2): tn, fp, fn, tp = cm.ravel()
    else: tn, fp, fn, tp = 0, 0, 0, 0

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    g_mean = np.sqrt(sensitivity * specificity)
    logloss = log_loss(y_true, y_pred_proba)
    auc = roc_auc_score(y_true, y_pred_proba)

    proba_pos = y_pred_proba[y_true == 1]
    proba_neg = y_pred_proba[y_true == 0]
    ks_stat, _ = ks_2samp(proba_pos, proba_neg) if len(proba_pos) > 0 and len(proba_neg) > 0 else (0.0, None)

    return {'AUC': auc, 'KS': ks_stat, 'F-measure': f_measure, 'G-Mean': g_mean, 'LogLoss': logloss}

def print_metrics_table(results_dict, verbose=True):
    if not verbose: return
    print("\n[8/11] Tabela de Desempenho (Métricas de Avaliação)...")
    df_metrics = pd.DataFrame.from_dict(results_dict, orient='index')
    for col in df_metrics.columns:
        df_metrics[col] = df_metrics[col].apply(lambda x: f"{x:.4f}")
    print("\n" + "="*70)
    print(df_metrics.to_string())
    print("="*70 + "\n")

def print_classification_reports(predictions_dict, y_true, le, verbose=True):
    if not verbose: return
    print("\n[9/11] Relatórios de Classificação Detalhados...")
    for model_name, y_pred in predictions_dict.items():
        print(f"\n{'='*60}\nModelo: {model_name}\n{'='*60}")
        print(classification_report(y_true, y_pred, target_names=le.classes_))

def plot_confusion_matrices(predictions_dict, y_true, le, verbose=True):
    if not verbose: return
    print("[10/11] Gerando Matrizes de Confusão Comparativas...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (model_name, y_pred) in zip(axes, predictions_dict.items()):
        cm = confusion_matrix(y_true, y_pred)
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens', xticklabels=le.classes_, yticklabels=le.classes_, ax=ax, cbar=False)
        ax.set_title(f'Matriz de Confusão\n{model_name}', fontsize=14, pad=15)
        ax.set_ylabel('Real', fontsize=12)
        ax.set_xlabel('Predito', fontsize=12)
    plt.tight_layout()
    plt.show()

def explain_student_risk(model, X_test, y_test, y_pred, y_pred_proba, base_feature_names, le, verbose=True):
    if not verbose: return
    print("\n[11/11] Analisando o Porquê da Evasão (SHAP Focado no Aluno)...")
    try:
        cancelado_idx = list(le.classes_).index('CANCELADO')
    except ValueError:
        print("    > Classe 'CANCELADO' não encontrada. Impossível rodar análise individual.")
        return

    evaded_indices = np.where((y_test == cancelado_idx) & (y_pred == cancelado_idx))[0]
    if len(evaded_indices) == 0:
        print("    > Nenhum aluno evadido encontrado no conjunto de teste para analisar.")
        return

    student_idx = evaded_indices[0]
    probabilidade_evasao = y_pred_proba[student_idx] if cancelado_idx == 1 else (1 - y_pred_proba[student_idx])

    X_student = X_test.tocsr()[student_idx:student_idx+1] if issparse(X_test) else X_test[student_idx:student_idx+1]
    X_student_last_layer = model.transform_to_last_layer(X_student)
    X_student_dense = X_student_last_layer.toarray() if issparse(X_student_last_layer) else X_student_last_layer

    final_estimator = model.layers_[-1][0] # Extrai do LightGBM para estabilidade com SHAP
    explainer = shap.TreeExplainer(final_estimator)
    shap_values = explainer.shap_values(X_student_dense)

    # Prevenção extra de Bugs do formato SHAP
    if isinstance(shap_values, list): shap_vals_student = shap_values[1][0]
    elif len(shap_values.shape) == 3: shap_vals_student = shap_values[0, :, 1]
    else: shap_vals_student = shap_values[0]

    # Prevenção contra a coluna fantasma (Base Value)
    if shap_vals_student.shape[0] == X_student_dense.shape[1] + 1:
        shap_vals_student = shap_vals_student[:-1]

    feature_names = model.get_feature_names_for_last_layer(base_feature_names)
    if len(feature_names) != X_student_dense.shape[1]:
        feature_names = feature_names[:X_student_dense.shape[1]]
        while len(feature_names) < X_student_dense.shape[1]: feature_names.append(f"Feature_Extra_{len(feature_names)}")

    df_exp = pd.DataFrame({
        'Disciplina_Variavel': feature_names,
        'Valor_Real_do_Aluno': X_student_dense[0],
        'Importancia_Absoluta': np.abs(shap_vals_student),
        'Peso_Shap': shap_vals_student
    }).sort_values(by='Importancia_Absoluta', ascending=False)

    print("\n" + "="*80)
    print(f"FICHA DE EXPLICABILIDADE - Aluno de Teste (Índice: {student_idx})")
    print(f"Risco Calculado de Evasão: {probabilidade_evasao * 100:.2f}%")
    print(f"Situação Real: {le.classes_[y_test[student_idx]]}")
    print("="*80)
    print("As 10 variáveis que mais impactaram a decisão deste aluno evadir:")
    for i, row in df_exp.head(10).iterrows():
        print(f"  -> {row['Disciplina_Variavel']:<35} | Valor lido pelo modelo: {row['Valor_Real_do_Aluno']:.4f}")
    print("="*80 + "\n")

# =====================================================================
# INFERÊNCIA / PRODUÇÃO - Função para prever em Arquivos Novos
# =====================================================================
def predict_new_students(new_filepath, model, preprocessor, expected_features, subject_names, verbose=True):
    if verbose: print(f"\n[SISTEMA EM PRODUÇÃO] Iniciando predição para novos alunos usando o arquivo: {new_filepath}")
    df = pd.read_csv(new_filepath)

    subject_cols = ['DESC_DISCIPLINA', 'CARGA_HOR', 'Sit_Disciplina', 'N_FALTAS', 'NOTA', 'Equivalencia']
    student_cols = [c for c in df.columns if c not in subject_cols]

    df_student = df[student_cols].drop_duplicates(subset=['Cod_Pessoa'])

    # Pivotar se houverem disciplinas (garante a arquitetura da floresta)
    if 'DESC_DISCIPLINA' in df.columns and 'NOTA' in df.columns:
        pivot_df = df.pivot_table(index='Cod_Pessoa', columns='DESC_DISCIPLINA', values='NOTA', aggfunc='max')
        df_merged = df_student.merge(pivot_df, on='Cod_Pessoa', how='left')
    else:
        df_merged = df_student.copy()

    # Ajuste perfeito das colunas conforme o modelo espera
    for col in expected_features:
        if col not in df_merged.columns:
            if col in subject_names:
                df_merged[col] = 0.0 # Se não tem a disciplina, nota zero
            else:
                df_merged[col] = np.nan # Deixa as variáveis cadastradas nulas para o pipeline imputar

    X_new = df_merged[expected_features]

    if verbose: print("      > Aplicando Pipeline (Imputação e Scaler) aos dados novos...")
    X_new_processed = preprocessor.transform(X_new)

    if verbose: print("      > Extraindo decisões profundas pela Floresta em Cascata...")
    probabilidades = model.predict_proba(X_new_processed)[:, 1]

    resultados = pd.DataFrame({
        'Cod_Pessoa': df_merged['Cod_Pessoa'],
        'Chance_Evasao_Perc': probabilidades * 100
    })

    # Regra de Negócio: Mais de 50% de chance classifica como Alto Risco
    resultados['Risco'] = np.where(resultados['Chance_Evasao_Perc'] > 50.0, 'ALTO RISCO (Evasão)', 'BAIXO RISCO (Conclusão)')

    if verbose:
        print("      > Predições Finalizadas! Aqui está uma amostra do parecer do Sistema de IA:")
        print(resultados.head(10).to_string(index=False))

    return resultados

# =====================================================================
# PIPELINE PRINCIPAL DE TREINAMENTO E VALIDAÇÃO
# =====================================================================
if __name__ == "__main__":
    df_clean, subject_names, leakage_cols = load_and_clean_data(verbose=VERBOSE)
    X_processed, y_encoded, le, preprocessor, expected_features = preprocess_sparse(df_clean, verbose=VERBOSE)

    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    if VERBOSE: print("\n[3/11] Iniciando PSO Multiparametrizado (T, Num_Leaves, Max_Depth)...")
    X_tr_pso, X_val_pso, y_tr_pso, y_val_pso = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

    # Limites das Partículas: T [1 a 8], Folhas [31 a 255], Profundidade [3 a 15]
    bounds = (np.array([1.0, 31.0, 3.0]), np.array([8.0, 255.0, 15.0]))
    optimizer = ps.single.GlobalBestPSO(n_particles=5, dimensions=3, options={'c1': 0.5, 'c2': 0.3, 'w': 0.9}, bounds=bounds)

    cost, pos = optimizer.optimize(optimize_hyperparams, iters=3, X_tr=X_tr_pso, y_tr=y_tr_pso, X_v=X_val_pso, y_v=y_val_pso, verbose=False)

    best_T = int(round(pos[0]))
    best_num_leaves = int(round(pos[1]))
    best_max_depth = int(round(pos[2]))

    if VERBOSE: print(f"\n[4/11] PSO Finalizado! Hiperparâmetros ideais: T={best_T} | Folhas_LGBM={best_num_leaves} | Depth_XGB={best_max_depth}")

    if VERBOSE: print("\n[5/11] Instanciando os Modelos Oficiais baseados nos achados do PSO...")

    # ------------------ HIPERPARÂMETROS SUBSTITUÍDOS ------------------
    lgbm_rf_base = LGBMClassifier(boosting_type='rf', num_leaves=best_num_leaves, bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8, n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
    xgb_rf_base = XGBRFClassifier(n_estimators=100, max_depth=best_max_depth, subsample=0.8, colsample_bynode=0.8, random_state=42, n_jobs=-1, eval_metric='logloss')
    ebca_deep_forest_model = EBCADeepCascadeForest(max_layers=5, k_fold=3, T=best_T, lgbm_num_leaves=best_num_leaves, xgb_max_depth=best_max_depth, random_state=42)
    # ------------------------------------------------------------------

    models_to_evaluate = {
        'LGBM RF (Isolado)': lgbm_rf_base,
        'XGBoost RF (Isolado)': xgb_rf_base,
        'EBCA + Deep Forest (PSO Otimizado)': ebca_deep_forest_model
    }

    results_metrics = {}
    predictions_dict = {}

    if VERBOSE: print("\n[6/11] Treinando modelos oficiais e realizando inferências no conjunto de teste...")
    for name, model in models_to_evaluate.items():
        if VERBOSE: print(f"      > Processando modelo: {name}...")

        if hasattr(model, 'max_layers'):
            model.fit(X_train, y_train, verbose=VERBOSE)
        else:
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        predictions_dict[name] = y_pred
        results_metrics[name] = calculate_paper_metrics(y_test, y_pred, y_pred_proba)

    print_metrics_table(results_metrics, verbose=VERBOSE)
    print_classification_reports(predictions_dict, y_test, le, verbose=VERBOSE)
    plot_confusion_matrices(predictions_dict, y_test, le, verbose=VERBOSE)

    base_feature_names = preprocessor.get_feature_names_out()
    explain_student_risk(ebca_deep_forest_model, X_test, y_test, predictions_dict['EBCA + Deep Forest (PSO Otimizado)'], ebca_deep_forest_model.predict_proba(X_test)[:, 1], base_feature_names, le, verbose=VERBOSE)

    if VERBOSE: print("\n[FIM DO TREINO] IA construída e otimizada com sucesso!")

    # =========================================================================
    # DEMONSTRAÇÃO DO SISTEMA FECHADO EM PRODUÇÃO (INFERÊNCIA PARA NOVOS DADOS)
    # =========================================================================
    # Para simular uma entrada real no sistema, você pode criar um novo csv
    # apenas com os alunos do ano atual (ou repassar a base inteira para testar)

    arquivo_novos_alunos = 'ifce_telecom_unificado - ifce_telecom_unificado.csv'
    df_previsoes_finais = predict_new_students(
        new_filepath=arquivo_novos_alunos,
        model=ebca_deep_forest_model,
        preprocessor=preprocessor,
        expected_features=expected_features,
        subject_names=subject_names,
        verbose=VERBOSE
    )

    # df_previsoes_finais.to_csv("Alunos_Em_Risco_Analise_IA.csv", index=False) # Descomente para salvar a lista de risco em Excel